# AI工学101 — 第31回

## データリークと再現性：機械学習実験を「正しく」する

よし、今日はかなり大事な回。

第30回で、

```text
性能が悪い
↓
なぜ？
↓
Underfitting？
Overfitting？
データ不足？
```

と**モデルを診断する**ところまで来た。

今日はさらに一段、工学寄りに進む。

> **「そもそも、その評価結果は信用していいのか？」**

という問題を扱う。

機械学習では、コードが動いてAccuracyが99%出ても、**実験の設計が間違っていれば、その99%には意味がない**。

特に危険なのが、

> **Data Leakage（データリーク）**

です。

---

# 🎯 今日のゴール

今日できるようになること：

* Data Leakageを説明できる
* Train/Test contaminationを見抜ける
* Target Leakageを説明できる
* 前処理によるリークを防げる
* Pipelineがなぜ重要なのか説明できる
* `random_state` の意味を理解する
* 再現可能な実験を設計できる
* 「高精度なのに本番で壊れる」理由を説明できる

---

# 📖 講義：約20〜25分

## 1. Data Leakageとは？

一言で言うと、

> **本来予測時には利用できない情報が、学習時にモデルへ入り込むこと**

です。

例えば、

```text
未来の情報
```

を使って、

```text
過去の出来事
```

を予測していたらおかしい。

---

# 💀 2. Target Leakage

例えば病院のデータで、

```text
年齢
血圧
血液検査
```

から、

```text
病気になるか？
```

を予測したいとします。

ところが特徴量の中に、

```text
退院時の診断結果
```

が入っていた。

これは、

```text
予測したい未来
```

の後に分かる情報です。

つまり、

```text
診断結果
 ↓
病気かどうか
```

がほぼ直接分かってしまう。

モデルは、

```text
99.9%
```

出すかもしれない。

でも本番では、

```text
退院時の診断結果
```

はまだ存在しません。

したがって、

**モデルは使えません。**

---

# 🧠 3. Target Leakageの本質

ここは単なる「testデータを混ぜた」問題ではありません。

重要なのは、

> **予測時点で利用可能だった情報だけを使う**

こと。

例えば、

```text
2026/8/20 10:00
```

に予測するなら、

```text
10:00以前に取得可能だった情報
```

だけを使う。

```text
10:05に発生した情報
```

を特徴量に入れたらアウト。

だから実務では、

> **時系列上の情報の利用可能時点**

も考えなければなりません。

---

# 💻 実習1：わざとリークを作る

人工データを作ります。

```python
import numpy as np
import pandas as pd

rng = np.random.RandomState(42)

n = 1000

df = pd.DataFrame({
    "age": rng.randint(20, 70, n),
    "income": rng.normal(500, 100, n),
})
```

ターゲット。

```python
df["target"] = (
    df["income"] > 550
).astype(int)
```

ここまでは、

```text
income
 ↓
target
```

という普通の予測問題です。

---

# 💻 実習2：リーク特徴量を作る

```python
df["leak"] = df["target"]
```

これで、

```text
leak
=
target
```

になりました。

当然、モデルはほぼ完璧に予測できます。

---

# 💻 実習3：モデルに入れてみる

```python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df[
    ["age", "income", "leak"]
]

y = df["target"]

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)
```

```python
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train,
    y_train
)
```

評価。

```python
print(
    model.score(
        X_test,
        y_test
    )
)
```

おそらく非常に高いAccuracyになります。

---

# 🚨 しかしこれは「高性能モデル」ではない

理由は単純。

```text
leak
 ↓
targetそのもの
```

だから。

モデルは、

> 「未来を予測している」

のではなく、

> **「答えを見ている」**

だけです。

---

# 💻 実習4：リークを除去

```python
X = df[
    ["age", "income"]
]
```

もう一度学習。

```python
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

model.fit(
    X_train,
    y_train
)

print(
    model.score(
        X_test,
        y_test
    )
)
```

今度は、

```text
leakあり
```

より性能が下がるはずです。

でもこちらのほうが、

**正しい評価**

です。

---

# 📖 4. 前処理によるData Leakage

次に、もっと現実的なリーク。

例えば、

```text
全データ
↓
StandardScaler
↓
train/test split
```

という順番にしてしまう。

これは危険です。

---

# 💀 なぜ？

StandardScalerは、

```text
平均
標準偏差
```

をデータから計算します。

もし全データを使って、

```python
scaler.fit_transform(X)
```

してから分割すると、

**testデータの平均・標準偏差を知った状態**

でtrainを前処理しています。

これは小さなリークです。

---

# ❌ 悪い例

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)

X_train, X_test, y_train, y_test = (
    train_test_split(
        X_scaled,
        y,
        test_size=0.2,
        random_state=42
    )
)
```

---

# ✅ 正しい方法

```text
まず分割
 ↓
trainでfit
 ↓
train/testをtransform
```

つまり、

```python
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)
```

---

# 🧠 ここで覚える鉄則

```text
TRAIN
 ↓
fit

TEST
 ↓
transform
```

testに対して、

```python
fit()
```

しない。

---

# 💡 そしてPipeline

第26回で、

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
```

を使った。

これが非常に重要です。

Pipelineにすると、

```text
train
 ↓
scaler.fit
 ↓
model.fit

test
 ↓
scaler.transform
 ↓
model.predict
```

という流れを安全に組みやすくなります。

---

# 💻 実習5：Pipelineで正しく評価

```python
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

```python
pipe.fit(
    X_train,
    y_train
)
```

```python
print(
    pipe.score(
        X_test,
        y_test
    )
)
```

これが第26回以降、ずっとPipelineを推している理由です。

---

# 🔥 5. Cross ValidationでもPipelineが重要

ここがさらに重要。

例えば、

```python
cross_val_score(
    pipe,
    X,
    y,
    cv=5
)
```

とすると、

概念的には、

```text
Fold 1

train
 ↓
Scaler.fit
 ↓
Model.fit
 ↓
validation.transform
 ↓
評価
```

となります。

Fold 2でも、

```text
別のtrain
 ↓
Scaler.fit
```

です。

つまり、

**各Foldのvalidationデータを前処理のfitに使わない。**

これが重要です。

---

# 💻 実習6：Pipelineなしの危険な例

わざと、

```python
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
```

してから、

```python
cross_val_score(
    LogisticRegression(
        max_iter=1000
    ),
    X_scaled,
    y,
    cv=5
)
```

をやってみます。

次に、

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

```python
cross_val_score(
    pipe,
    X,
    y,
    cv=5
)
```

を比較します。

差が小さいこともあります。

でも重要なのは、

> **結果がたまたま近いかどうかではなく、実験手順が正しいか**

です。

---

# 📖 6. `random_state` とは？

次は再現性。

例えば、

```python
train_test_split(
    X,
    y,
    test_size=0.2
)
```

を実行すると、

毎回同じ分割になるとは限りません。

そこで、

```python
random_state=42
```

を指定します。

---

# 💻 実習7：random_stateの違い

```python
X_train1, X_test1, y_train1, y_test1 = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)
```

もう一度、

```python
X_train2, X_test2, y_train2, y_test2 = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)
```

比較。

```python
print(
    np.array_equal(
        X_train1,
        X_train2
    )
)
```

`True` になるはずです。

---

# 💻 実習8：seedを変える

```python
X_train3, X_test3, y_train3, y_test3 = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=123
    )
)
```

今度は、

```python
print(
    np.array_equal(
        X_train1,
        X_train3
    )
)
```

おそらく `False`。

つまり、

```text
random_state=42
```

と、

```text
random_state=123
```

では別の実験条件です。

---

# 🧠 7. 再現性とは？

再現性とは、

> **同じ条件で実験したとき、同じ結果を再現できること**

です。

例えば、

```text
データ
モデル
ハイパーパラメータ
train/test split
乱数seed
評価方法
```

を記録しておけば、

別の日でも、

別の環境でも、

**同じ実験を再現しやすくなります。**

---

# ⚠️ `random_state=42` は魔法ではない

ここは勘違いしないように。

```python
random_state=42
```

そのものに特別な意味があるわけではありません。

42でも、

```text
0
1
123
2026
```

でも構いません。

重要なのは、

> **実験条件を固定して記録すること**

です。

---

# 💻 実習9：モデル側のrandom_state

Random Forestにも、

```python
RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
```

と指定できます。

これは、

**モデル内部の乱数を固定する**

ためです。

つまり、

```text
データ分割のrandom_state
```

と、

```text
モデルのrandom_state
```

は別のもの。

---

# 🧠 8. 「再現性」と「頑健性」は違う

ここは一歩進んだ話。

例えば、

```text
seed=42
Accuracy=0.95
```

が再現できても、

```text
seed=123
Accuracy=0.82
```

だったら、

> **結果が分割にかなり依存している**

可能性があります。

つまり、

```text
再現可能
```

と、

```text
どんな分割でも安定
```

は別です。

後者を見るには、

```text
Cross Validation
Repeated CV
複数seed
```

などが役立ちます。

---

# 💻 実習10：複数seedを試す

```python
seeds = [
    0,
    1,
    2,
    42,
    123
]
```

```python
scores = []

for seed in seeds:

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=seed,
            stratify=y
        )
    )

    pipe.fit(
        X_train,
        y_train
    )

    score = pipe.score(
        X_test,
        y_test
    )

    scores.append(
        score
    )
```

確認。

```python
print(
    scores
)

print(
    np.mean(scores)
)

print(
    np.std(scores)
)
```

これで、

```text
平均性能
ばらつき
```

を見ることができます。

---

# 🧠 9. 本番環境を想像する

ここで重要な思考実験。

開発時には、

```text
Accuracy = 99%
```

だった。

しかし本番では、

```text
Accuracy = 72%
```

になった。

何が考えられる？

例えば、

```text
① Data Leakage

② Train/Testの分布が違う

③ 時系列の変化

④ 本番データの品質が違う

⑤ 特徴量の取得方法が違う

⑥ 欠損処理が違う

⑦ 本番では利用できない情報を学習時に使っていた
```

など。

つまり、

> **機械学習モデルの評価は、単なるスコア計算ではない。**

---

# 🧠 10. Train/Test Distribution Shift

例えば、

```text
train：
2024〜2025年

test：
2026年
```

だとします。

時間とともに、

```text
ユーザー行動
市場
社会
センサー
```

が変化しているかもしれません。

この場合、

```text
IID
```

という単純な前提が崩れる可能性があります。

これは今後、

**時系列データ・Distribution Shift**

へ進むための重要な入口です。

---

# ✍️ 演習

## 問1

次の処理のどこが問題でしょう？

```python
scaler.fit_transform(X)

train_test_split(...)
```

---

## 問2

なぜ、

```python
scaler.fit(X_train)
```

はOKで、

```python
scaler.fit(X_test)
```

は避けるのでしょうか？

---

## 問3

次の特徴量はTarget Leakageになり得るでしょうか？

```text
「ローン審査結果」を予測するモデル

特徴量：
年齢
年収
勤続年数
最終審査結果
```

---

## 問4

```python
random_state=42
```

にはどんな意味がありますか？

---

## 問5

次の2つは同じでしょうか？

```text
再現性が高い
```

と、

```text
モデルが頑健
```

理由も説明してください。

---

# 👾 ボス戦：リークを見抜け

次の機械学習システムを考えます。

```text
目的：
明日の顧客離脱を予測する
```

特徴量：

```text
年齢
契約期間
過去30日間のログイン回数
過去30日間の問い合わせ数
本日までの利用時間
解約処理完了日時
```

この中に、

> **予測時点では利用できない可能性がある情報**

があります。

どれでしょう？

そして、

```text
「なぜリークになり得るのか」
```

を時系列で説明してください。

---

# 🧪 最終実習：正しいML実験パイプライン

今日の内容を全部つなぎます。

最終的に目指す構造はこれ。

```text
生データ
   ↓
train / test split
   ↓
        TRAIN
          ↓
   Pipeline + CV
          ↓
  前処理をfit
          ↓
  モデルをfit
          ↓
  Hyperparameter Search
          ↓
    最良モデル
          ↓
        TEST
          ↓
     最終評価
```

そして実験条件として、

```text
random_state
モデル
前処理
評価指標
CV設定
探索範囲
```

を記録します。

---

# 🌱 今日のまとめ

今日の一番大事なこと。

> **機械学習では、「高いスコアを出す」より先に「そのスコアを信じていい実験になっているか」を確認する。**

覚えるべき原則はこれ。

```text
① testは最後まで温存

② 前処理はtrainでfit

③ CVの中でも前処理を分離

④ Pipelineを使う

⑤ 予測時点で利用可能な情報だけ使う

⑥ random_stateなど実験条件を記録

⑦ 1回のsplitだけで性能を断定しない
```

特に、

```text
fit
```

と

```text
transform
```

の違いは、これからPyTorchまでずっと出てくる。

---

# 🧭 AI工学101・現在地

ここまでで、scikit-learn編はかなり完成形に近づいてきた。

```text
Python / NumPy
 ↓
データ操作
 ↓
回帰・分類
 ↓
評価指標
 ↓
前処理
 ↓
特徴量
 ↓
Feature Selection / PCA
 ↓
モデル
 ↓
Cross Validation
 ↓
Hyperparameter Search
 ↓
Learning / Validation Curve
 ↓
Model Interpretation
 ↓
Data Leakage
 ↓
Reproducibility
```

そして最終的には、

```text
データ
 ↓
仮説
 ↓
前処理
 ↓
モデル
 ↓
評価
 ↓
診断
 ↓
改善
 ↓
再評価
```

という**機械学習開発の実験ループ**を、自分で設計できるところまで来ています。

これは「scikit-learnの使い方」を覚えたというより、**機械学習を工学的に扱うための型**を身につけ始めた、という段階だね。

---

# 🔜 第32回

## 時系列データと交差検証：未来の情報を混ぜない

次はData Leakageをさらに実戦的にします。

普通の、

```text
train_test_split
```

ではなく、

```text
時間
 ↓
過去 → 未来
```

という構造を持つデータを扱います。

内容は、

* Time Series Split
* なぜランダムシャッフルが危険なのか
* Walk-forward validation
* 過去で学習 → 未来で評価
* 時系列におけるData Leakage
* Concept Drift / Distribution Shiftの入口
* 株価・需要予測・ログデータなどへの応用

です。

ここを終えると、**「データをランダムに混ぜてはいけない問題」**も自分で設計できるようになります。